In [34]:
import random
import networkx as nx
import json
from pprint import pprint

def generate_job_scheduler_task(num_tasks=10, edge_probability=0.6):
    # 1. Generate a random Directed Acyclic Graph (DAG)
    G = nx.DiGraph()
    
    # Create task names
    task_names = [f"Task_{chr(65+i)}" for i in range(num_tasks)] # Task_A, Task_B, etc.
    G.add_nodes_from(task_names)
    
    # Randomly add dependencies (ensuring it flows in one direction to prevent cycles)
    for i in range(num_tasks):
        for j in range(i + 1, num_tasks):
            if random.random() < edge_probability:
                G.add_edge(task_names[i], task_names[j])
                
    # 2. Get the Ground Truth Solution
    # networkx has a built-in topological sort to find a valid order
    try:
        valid_sequence = list(nx.topological_sort(G))
    except nx.NetworkXUnfeasible:
        # Failsafe if a cycle somehow exists
        return None

    # 3. Generate the Text Prompt
    dependencies_text = []
    for u, v in G.edges():
        dependencies_text.append(f"- {u} must be completed before starting {v}.")
    
    if not dependencies_text:
        dependencies_text.append("- There are no dependencies. Tasks can be done in any order.")
        
    prompt = (
        f"You are a project manager. You have {num_tasks} tasks to complete: {', '.join(task_names)}.\n"
        "Here are the strict dependency rules you must follow:\n"
        f"{chr(10).join(dependencies_text)}\n\n"
        "Provide a valid sequence to complete all tasks from first to last. "
        "Output ONLY the sequence as a comma-separated list."
    )
    
    # 4. Return the complete dataset row
    return {
        "prompt": prompt,
        "ground_truth_sequence": valid_sequence,
        "graph_edges": list(G.edges())
    }

# Generate 3 examples for your dataset
dataset = []
for _ in range(3):
    task = generate_job_scheduler_task(num_tasks=6, edge_probability=0.2)
    if task:
        dataset.append(task)

# Print the first example to see what it looks like
pprint(json.dumps(dataset[0], indent=2))

('{\n'
 '  "prompt": "You are a project manager. You have 6 tasks to complete: '
 'Task_A, Task_B, Task_C, Task_D, Task_E, Task_F.\\nHere are the strict '
 'dependency rules you must follow:\\n- Task_A must be completed before '
 'starting Task_C.\\n- Task_B must be completed before starting Task_F.\\n- '
 'Task_C must be completed before starting Task_E.\\n- Task_D must be '
 'completed before starting Task_E.\\n- Task_E must be completed before '
 'starting Task_F.\\n\\nProvide a valid sequence to complete all tasks from '
 'first to last. Output ONLY the sequence as a comma-separated list.",\n'
 '  "ground_truth_sequence": [\n'
 '    "Task_A",\n'
 '    "Task_B",\n'
 '    "Task_D",\n'
 '    "Task_C",\n'
 '    "Task_E",\n'
 '    "Task_F"\n'
 '  ],\n'
 '  "graph_edges": [\n'
 '    [\n'
 '      "Task_A",\n'
 '      "Task_C"\n'
 '    ],\n'
 '    [\n'
 '      "Task_B",\n'
 '      "Task_F"\n'
 '    ],\n'
 '    [\n'
 '      "Task_C",\n'
 '      "Task_E"\n'
 '    ],\n'
 '    [\n'
 '      "Ta

In [305]:
import json
import random
from collections import deque

# ==========================================
# 1. SCENARIO GENERATOR (Deep Graph Logic)
# ==========================================
def generate_deep_scenario(num_nodes=15):
    terrains = ["Normal", "Normal", "Desert", "Rocky", "Magnetic"]
    
    item_pool = [
        {"name": "Ice", "weight": 2},
        {"name": "Glass", "weight": 1},
        {"name": "Hard Drive", "weight": 1},
        {"name": "Heavy Generator", "weight": 5},
        {"name": "Cooler", "weight": 2},       
        {"name": "Lead Shield", "weight": 3},  
    ]

    # Generate Inventory
    num_items = random.randint(3, 5)
    inventory = random.sample(item_pool, num_items)
    total_weight = sum(item['weight'] for item in inventory)
    vehicle_max_weight = total_weight + random.randint(0, 3)

    # Generate Nodes in a strict topological order to allow deep paths
    start_node = "Base"
    end_node = "Destination"
    intermediate_nodes = [f"Node_{i}" for i in range(1, num_nodes + 1)]
    all_nodes = [start_node] + intermediate_nodes + [end_node]
    
    graph = {node: {} for node in all_nodes}
    total_nodes = len(all_nodes)

    # We connect nodes to the next few nodes in the list. This stretches the graph out.
    for i in range(total_nodes - 1):
        u = all_nodes[i]
        
        # Determine how far forward this node can connect
        # Prefer connecting to immediate neighbors to force long paths
        max_reach = min(i + 4, total_nodes) 
        possible_targets = list(range(i + 1, max_reach))
        
        # Add a 33% chance for a "Mirage Path" (a long shortcut that might be a trap)
        if i + 4 < total_nodes and random.random() < 0.33:
            possible_targets.append(random.randint(i + 4, total_nodes - 1))
            
        # Pick 1 to 3 random targets from the valid pool
        num_connections = random.randint(1, min(3, len(possible_targets)))
        targets = random.sample(possible_targets, num_connections)
        
        for j in targets:
            v = all_nodes[j]
            graph[u][v] = {
                "time": random.randint(1, 5),
                "terrain": random.choice(terrains)
            }

    # Orphan Catcher: Ensure every node is reachable
    for i in range(1, total_nodes):
        v = all_nodes[i]
        if not any(v in graph[all_nodes[k]] for k in range(i)):
            # Force connection from the immediately preceding node
            graph[all_nodes[i-1]][v] = {"time": random.randint(1, 5), "terrain": random.choice(terrains)}

    # Dead-end Catcher: Ensure every node can reach the destination
    for i in range(total_nodes - 1):
        u = all_nodes[i]
        if not graph[u]:
            # Force connection to the immediately next node
            graph[u][all_nodes[i+1]] = {"time": random.randint(1, 5), "terrain": random.choice(terrains)}

    return {
        "graph": graph,
        "start_node": start_node,
        "end_node": end_node,
        "inventory": inventory,
        "vehicle_max_weight": vehicle_max_weight
    }

# ==========================================
# 2. STATE-TRACKING BFS SOLVER
# ==========================================
def solve_logistics_puzzle(scenario):
    graph = scenario['graph']
    start_node = scenario['start_node']
    end_node = scenario['end_node']
    inventory = scenario['inventory']
    item_names = [item['name'] for item in inventory]
    
    if sum(item['weight'] for item in inventory) > scenario['vehicle_max_weight']:
        return {"status": "Failed"}
    
    # Queue: (current_node, current_time, path_history, deserts_crossed)
    # We MUST track deserts_crossed in the state to calculate Cooler degradation
    queue = deque([(start_node, 0, [start_node], 0)])
    
    # Visited dictionary keyed by (Node, State) to allow backtracking if a cooler breaks
    shortest_time_to_node = {(start_node, 0): 0}
    
    best_path = None
    best_time = float('inf')

    while queue:
        current_node, current_time, path, deserts_crossed = queue.popleft()
        
        if current_node == end_node:
            if current_time < best_time:
                best_time = current_time
                best_path = path
            continue
            
        for neighbor, edge_data in graph[current_node].items():
            terrain = edge_data['terrain']
            base_time = edge_data['time']
            new_deserts = deserts_crossed + 1 if terrain == "Desert" else deserts_crossed
            
            # HARDCORE RULES ENGINE
            if terrain == "Desert" and "Ice" in item_names:
                if "Cooler" not in item_names:
                    continue # Melts instantly
                if new_deserts > 1:
                    continue # Cooler ran out of battery!
                    
            if terrain == "Rocky" and "Glass" in item_names:
                continue
            if terrain == "Magnetic" and "Hard Drive" in item_names and "Lead Shield" not in item_names:
                continue
                
            move_time = base_time + 1 if "Heavy Generator" in item_names else base_time
            new_time = current_time + move_time
            
            # State-based pruning
            state_key = (neighbor, new_deserts)
            if state_key not in shortest_time_to_node or new_time < shortest_time_to_node[state_key]:
                shortest_time_to_node[state_key] = new_time
                queue.append((neighbor, new_time, path + [neighbor], new_deserts))

    if best_path:
        return {"status": "Success", "optimal_time": best_time, "optimal_path": best_path}
    return {"status": "Failed"}

# ==========================================
# 3. PROMPT GENERATOR
# ==========================================
def generate_prompt(scenario):
    inventory_text = [f"- {item['name']} (Weight: {item['weight']})" for item in scenario['inventory']]
    map_text = []
    for node, edges in scenario['graph'].items():
        if not edges: continue
        connections = [f"{dest} (Time: {details['time']} hrs, Terrain: {details['terrain']})" for dest, details in edges.items()]
        map_text.append(f"From {node}, you can travel to: {', '.join(connections)}.")
        
    prompt = (
        f"You are navigating a transport vehicle from '{scenario['start_node']}' to '{scenario['end_node']}'. "
        f"Find the fastest valid route without destroying any cargo.\n\n"
        f"### YOUR VEHICLE & INVENTORY\n"
        f"Maximum Vehicle Weight Capacity: {scenario['vehicle_max_weight']}\n"
        f"Current Inventory:\n{chr(10).join(inventory_text)}\n\n"
        f"### THE MAP\n{chr(10).join(map_text)}\n\n"
        f"### THE RULES\n"
        f"1. You cannot exceed the maximum vehicle weight capacity at any time.\n"
        f"2. If you carry 'Glass' through a 'Rocky' terrain, it will shatter.\n"
        f"3. If you carry a 'Hard Drive' through a 'Magnetic' terrain, it will wipe out UNLESS you carry a 'Lead Shield'.\n"
        f"4. If you carry 'Ice' through a 'Desert' terrain, it will melt UNLESS you also carry a 'Cooler'.\n"
        f"5. STATE DEGRADATION: Your 'Cooler' only has enough battery to keep the ice intact for exactly ONE 'Desert' terrain. If you enter a second 'Desert' terrain, the Ice will melt.\n"
        f"6. DISTRACTOR RULE: 'Biological Samples' spoil in 'Normal' terrain after 3 hours. (Ensure you are actually carrying these before applying this rule).\n"
        f"7. TIME PENALTY: Carrying the 'Heavy Generator' adds +1 hour to EVERY path you take.\n\n"
        f"Provide the absolute fastest valid path with least total time. Output ONLY a valid JSON object with exactly two keys: "
        f"'path' (a list of exact node names) and 'total_time' (an integer representing the total hours)."
    )
    return prompt

# ==========================================
# 4. DATASET GENERATOR LOOP
# ==========================================
def create_dataset(target_size=2):
    dataset = []
    attempts = 0
    print(f"Generating deep scenarios...")
    
    while len(dataset) < target_size:
        attempts += 1
        # Generate a massive 15-node graph
        number_of_nodes = 15
        scenario = generate_deep_scenario(num_nodes=number_of_nodes)
        solution = solve_logistics_puzzle(scenario)
        deep_path_length = number_of_nodes / 1.5 + 1
        
        if solution["status"] == "Success":
            path = solution["optimal_path"]
            
            # THE FIX: Iterate through the edges to count Normal terrains
            normal_count = 0
            for i in range(len(path) - 1):
                u = path[i]
                v = path[i+1]
                if scenario['graph'][u][v]['terrain'] == "Normal":
                    normal_count += 1
            
            # Check if strictly more than half the edges are Normal
            num_edges = len(path) - 1
            normal_more_than_half = normal_count > max(7, (num_edges / 1.5))
            
            if len(path) > deep_path_length and not normal_more_than_half:
                dataset_row = {
                    "id": f"fragile_inventory_puzzle_{len(dataset) + 1:03d}",
                    "prompt": generate_prompt(scenario),
                    "expected_output": {
                        "path": path,
                        "total_time": solution["optimal_time"],
                        "path_length": len(path)
                    },
                    "scenario_details": scenario 
                }
                dataset.append(dataset_row)
            
    print(f"Generated {target_size} deep scenarios out of {attempts} attempts.")
    return dataset
data = create_dataset(target_size=1)
pprint(data)

Generating deep scenarios...
Generated 1 deep scenarios out of 7591 attempts.
[{'expected_output': {'path': ['Base',
                               'Node_1',
                               'Node_2',
                               'Node_4',
                               'Node_5',
                               'Node_7',
                               'Node_8',
                               'Node_10',
                               'Node_11',
                               'Node_14',
                               'Node_15',
                               'Destination'],
                      'path_length': 12,
                      'total_time': 28},
  'id': 'fragile_inventory_puzzle_001',
  'prompt': "You are navigating a transport vehicle from 'Base' to "
            "'Destination'. Find the fastest valid route without destroying "
            'any cargo.\n'
            '\n'
            '### YOUR VEHICLE & INVENTORY\n'
            'Maximum Vehicle Weight Capacity: 8\n'
            '

In [306]:
#copy prompt to clipboard
from pyperclip import copy
copy(data[0]["prompt"])

print(data[0]["prompt"])

You are navigating a transport vehicle from 'Base' to 'Destination'. Find the fastest valid route without destroying any cargo.

### YOUR VEHICLE & INVENTORY
Maximum Vehicle Weight Capacity: 8
Current Inventory:
- Cooler (Weight: 2)
- Hard Drive (Weight: 1)
- Ice (Weight: 2)

### THE MAP
From Base, you can travel to: Node_1 (Time: 4 hrs, Terrain: Rocky).
From Node_1, you can travel to: Node_3 (Time: 5 hrs, Terrain: Magnetic), Node_2 (Time: 1 hrs, Terrain: Rocky).
From Node_2, you can travel to: Node_4 (Time: 1 hrs, Terrain: Normal), Node_5 (Time: 5 hrs, Terrain: Desert).
From Node_3, you can travel to: Node_5 (Time: 3 hrs, Terrain: Rocky).
From Node_4, you can travel to: Node_5 (Time: 1 hrs, Terrain: Desert).
From Node_5, you can travel to: Node_8 (Time: 4 hrs, Terrain: Rocky), Node_7 (Time: 1 hrs, Terrain: Rocky), Node_6 (Time: 1 hrs, Terrain: Magnetic).
From Node_6, you can travel to: Node_7 (Time: 5 hrs, Terrain: Normal), Node_9 (Time: 1 hrs, Terrain: Rocky).
From Node_7, you can tr

In [255]:
def validate_llm_path(scenario, proposed_path):
    # 1. Basic Type and Structure Checks
    if not proposed_path or not isinstance(proposed_path, list):
        return {"is_valid": False, "reason": "Path is empty or not a list."}
        
    start_node = scenario['start_node']
    end_node = scenario['end_node']
    
    if proposed_path[0] != start_node:
        return {"is_valid": False, "reason": f"Path must start at {start_node}."}
    if proposed_path[-1] != end_node:
        return {"is_valid": False, "reason": f"Path must end at {end_node}."}

    # 2. Extract Inventory State
    inventory = scenario['inventory']
    item_names = [item['name'] for item in inventory]
    graph = scenario['graph']
    
    # 3. Track Runtime State
    deserts_crossed = 0
    calculated_time = 0
    has_heavy_generator = "Heavy Generator" in item_names

    # 4. Walk the LLM's Path Step-by-Step
    for i in range(len(proposed_path) - 1):
        current_node = proposed_path[i]
        next_node = proposed_path[i+1]
        
        # Check if the path actually exists in the map (No Teleporting!)
        if current_node not in graph or next_node not in graph[current_node]:
            return {
                "is_valid": False, 
                "reason": f"Hallucinated path: No road exists from {current_node} to {next_node}."
            }
            
        edge_data = graph[current_node][next_node]
        terrain = edge_data['terrain']
        base_time = edge_data['time']
        
        # Apply Time Penalty
        move_time = base_time + 1 if has_heavy_generator else base_time
        calculated_time += move_time
        
        # Apply Rules Engine to the specific terrain
        if terrain == "Desert":
            deserts_crossed += 1
            if "Ice" in item_names:
                if "Cooler" not in item_names:
                    return {"is_valid": False, "reason": f"Ice melted at {next_node} (Desert terrain, no Cooler)."}
                if deserts_crossed > 1:
                    return {"is_valid": False, "reason": f"Ice melted at {next_node} (Cooler battery died on 2nd Desert)."}
                    
        elif terrain == "Rocky":
            if "Glass" in item_names:
                return {"is_valid": False, "reason": f"Glass shattered at {next_node} (Rocky terrain)."}
                
        elif terrain == "Magnetic":
            if "Hard Drive" in item_names and "Lead Shield" not in item_names:
                return {"is_valid": False, "reason": f"Hard Drive wiped at {next_node} (Magnetic terrain, no Shield)."}

    # 5. If it survives the loop, the path is completely valid!
    return {
        "is_valid": True,
        "calculated_time": calculated_time,
        "path_length": len(proposed_path)
    }

# --- EXAMPLE USAGE FOR SCORING ---


llm_response_1 = ["Base", "Node_1", "Node_2", "Node_4", "Node_6", "Node_8", "Node_10", "Node_13", "Node_15", "Destination"],
llm_response_2 = ["Base", "Node_1", "Node_2", "Node_3", "Node_6", "Node_7", "Node_8", "Node_11", "Node_13", "Node_14", "Node_15", "Destination"]
                  
true = ['Base',
                               'Node_1',
                               'Node_2',
                               'Node_4',
                               'Node_7',
                               'Node_8',
                               'Node_9',
                               'Node_10',
                               'Node_13',
                               'Node_14',
                               'Node_15',
                               'Destination']
# You would run:
current_scenario = data[0]['scenario_details']  # The scenario we generated and gave to the LLM
result_1 = validate_llm_path(current_scenario, llm_response_2)
print(result_1) 

# Output: {'is_valid': True, 'calculated_time': 36, 'path_length': 8}

{'is_valid': False, 'reason': 'Ice melted at Node_11 (Desert terrain, no Cooler).'}


In [285]:
import random
import math
import pandas as pd

def generate_spatial_drift(num_steps=5):
    # Define movement logic on a Cartesian plane
    directions = {
        'North': (0, 1), 
        'South': (0, -1),
        'East': (1, 0), 
        'West': (-1, 0)
    }
    
    # Starting coordinates
    true_x, true_y = 0, 0
    trap_x, trap_y = 0, 0

    error_types = ["ShiftType", "OrthogonalType"]
    
    # Generate the random sequence of moves
    steps_data = []
    for _ in range(num_steps):
        direction = random.choice(list(directions.keys()))
        magnitude = random.randint(3, 12)
        steps_data.append((direction, magnitude))
        
    # Pick a random step in the middle to inject the Mathematical Drift
    # We avoid the first step and the very last step
    error_step_index = random.randint(1, num_steps - 2) 
    
    prompt_lines = [
        "I am tracking the exact coordinates of a drone on a 2D Cartesian plane. It starts at (0, 0)."
    ]
    
    for i, (direction, magnitude) in enumerate(steps_data):
        step_num = i + 1
        dx, dy = directions[direction]
        
        # 1. Update the TRUE background state
        true_x += dx * magnitude
        true_y += dy * magnitude
        
        # 2. Update the TRAP state (what the model sees)
        if i == error_step_index:
            error_type = random.choice(error_types)

            if (error_type == "ShiftType"):

                # INJECT THE ERROR: Shift the calculation by a small margin (+4 or -4)
                error_offset = random.choice([-4, -3, -2, 2, 3, 4])
                
                if dx != 0: # If moving East/West, mess up the X coordinate calculation
                    trap_x = trap_x + (dx * magnitude) + error_offset
                    trap_y = trap_y + (dy * magnitude)
                else:       # If moving North/South, mess up the Y coordinate calculation
                    trap_x = trap_x + (dx * magnitude)
                    trap_y = trap_y + (dy * magnitude) + error_offset
            elif (error_type == "OrthogonalType"):
                # INJECT THE ERROR: Swap the direction of movement (e.g., North becomes East)
                orthogonal_dx, orthogonal_dy = -dy, dx  # Rotate 90 degrees
                trap_x += orthogonal_dx * magnitude
                trap_y += orthogonal_dy * magnitude
                
        else:
            # Normal calculation based on the current trap state
            trap_x += dx * magnitude
            trap_y += dy * magnitude
            
        # 3. Format the text for the prompt
        if i < num_steps - 1:
            prompt_lines.append(f"Step {step_num}: The drone flies {magnitude} units {direction}. Current position: ({trap_x}, {trap_y}).")
        else:
            # Leave the final step open for the LLM to solve
            prompt_lines.append(f"Step {step_num}: The drone flies {magnitude} units {direction}...")
            
    # Add the final instruction requiring heavy computation (Hypotenuse)
    prompt_lines.append("\nBased on this flight path, please finish the final step to find the final coordinates.")
    prompt_lines.append("Then, calculate the exact direct hypotenuse distance from the origin (0, 0) to the final position using the distance formula. Round your final distance to two decimal places and output the final answer strictly in <distance> ... </distance> format.")
    
    prompt = "\n".join(prompt_lines)
    
    # Calculate final distances for grading
    true_distance = round(math.hypot(true_x, true_y), 2)
    trap_distance = round(math.hypot(trap_x, trap_y), 2)
    
    return {
        "prompt": prompt,
        "true_final_coords": f"({true_x}, {true_y})",
        "trap_final_coords": f"({trap_x}, {trap_y})",
        "true_distance": true_distance,
        "trap_distance": trap_distance,
        "error_step": error_step_index + 1,
        "error_type": error_type
    }

# --- Generate a Bulk Dataset ---
dataset = []
for _ in range(5): # Change to 5000 for your actual Kaggle generation
    dataset.append(generate_spatial_drift(num_steps=12))

# Convert to Pandas DataFrame for easy viewing/saving
df = pd.DataFrame(dataset)

# Display an example
sample = df.iloc[0]
print("--- GENERATED PROMPT ---")
print(sample['prompt'])
print("\n--- GRADING KEY ---")
print(f"Error Injected at Step: {sample['error_step']} (Error Type: {sample['error_type']})")
print(f"True Target Distance: {sample['true_distance']} (Model has Metacognition)")
print(f"Trap Target Distance: {sample['trap_distance']} (Model failed/hallucinated)")
copy(sample['prompt'])

--- GENERATED PROMPT ---
I am tracking the exact coordinates of a drone on a 2D Cartesian plane. It starts at (0, 0).
Step 1: The drone flies 4 units West. Current position: (-4, 0).
Step 2: The drone flies 12 units East. Current position: (8, 0).
Step 3: The drone flies 12 units South. Current position: (8, -12).
Step 4: The drone flies 8 units South. Current position: (8, -20).
Step 5: The drone flies 10 units East. Current position: (20, -20).
Step 6: The drone flies 11 units North. Current position: (20, -9).
Step 7: The drone flies 5 units East. Current position: (25, -9).
Step 8: The drone flies 11 units West. Current position: (14, -9).
Step 9: The drone flies 3 units West. Current position: (11, -9).
Step 10: The drone flies 12 units North. Current position: (11, 3).
Step 11: The drone flies 7 units East. Current position: (18, 3).
Step 12: The drone flies 12 units East...

Based on this flight path, please finish the final step to find the final coordinates.
Then, calculate th

In [146]:
import itertools
from collections import deque
from pprint import pprint

import itertools
from collections import deque

def solve_knapsack_logistics_puzzle(scenario):
    """
    Calculates the optimal path and loadout for the Logistics Knapsack benchmark.
    Tests all valid combinations of optional items against the map topology.
    """
    graph = scenario['graph']
    start_node = scenario['start_node']
    end_node = scenario['end_node']
    
    required_cargo = scenario['required_cargo']
    optional_protections = scenario['optional_protections']
    max_weight = scenario['vehicle_max_weight']
    
    # 1. Calculate base weight of required items
    base_weight = sum(item['weight'] for item in required_cargo)
    if base_weight > max_weight:
        return {"status": "Failed", "reason": "Required cargo exceeds max weight."}
        
    remaining_capacity = max_weight - base_weight
    
    # 2. Generate all valid Knapsack loadouts
    valid_loadouts = []
    for i in range(len(optional_protections) + 1):
        for combo in itertools.combinations(optional_protections, i):
            if sum(item['weight'] for item in combo) <= remaining_capacity:
                valid_loadouts.append(list(combo))
                
    if not valid_loadouts:
        valid_loadouts.append([])

    global_best_time = float('inf')
    global_best_path = None
    global_best_inventory = None

    # 3. Run the BFS Maze Solver for EVERY valid loadout
    for optional_loadout in valid_loadouts:
        current_inventory = required_cargo + optional_loadout
        item_names = [item['name'] for item in current_inventory]
        
        # ADDED: State now tracks `normal_time` as well
        # State: (node, total_time, path, deserts_crossed, normal_time)
        queue = deque([(start_node, 0, [start_node], 0, 0)])
        
        # ADDED: State key must include normal_time if we are carrying Biological Samples
        shortest_time_to_node = {(start_node, 0, 0): 0}
        
        best_time_for_loadout = float('inf')
        best_path_for_loadout = None

        while queue:
            current_node, current_time, path, deserts_crossed, normal_time = queue.popleft()
            
            if current_node == end_node:
                if current_time < best_time_for_loadout:
                    best_time_for_loadout = current_time
                    best_path_for_loadout = path
                continue
                
            for neighbor, edge_data in graph[current_node].items():
                terrain = edge_data['terrain']
                base_time = edge_data['time']
                
                # Apply the heavy generator penalty
                move_time = base_time + 1 if "Heavy Generator" in item_names else base_time
                new_time = current_time + move_time
                
                # Track hidden states
                new_deserts = deserts_crossed + 1 if terrain == "Desert" else deserts_crossed
                new_normal_time = normal_time + move_time if terrain == "Normal" else normal_time
                
                # --- RULES ENGINE ---
                if terrain == "Desert" and "Ice" in item_names:
                    if "Cooler" not in item_names or new_deserts > 1:
                        continue # Ice melts
                        
                if terrain == "Rocky" and "Glass" in item_names:
                    continue # Glass shatters
                    
                if terrain == "Magnetic" and "Hard Drive" in item_names:
                    if "Lead Shield" not in item_names:
                        continue # Hard Drive wiped
                
                # ADDED: Biological Samples Constraint
                if "Biological Samples" in item_names and new_normal_time > 3:
                    continue # Biological Samples spoiled!
                    
                # State-based pruning
                # If carrying bio samples, the amount of normal time used matters for future moves
                state_key = (neighbor, new_deserts, new_normal_time if "Biological Samples" in item_names else 0)
                
                if state_key not in shortest_time_to_node or new_time < shortest_time_to_node[state_key]:
                    shortest_time_to_node[state_key] = new_time
                    queue.append((neighbor, new_time, path + [neighbor], new_deserts, new_normal_time))

        if best_time_for_loadout < global_best_time:
            global_best_time = best_time_for_loadout
            global_best_path = best_path_for_loadout
            global_best_inventory = item_names

    if global_best_path:
        return {
            "status": "Success", 
            "optimal_time": global_best_time, 
            "optimal_path": global_best_path,
            "optimal_inventory": global_best_inventory
        }
    return {"status": "Failed"}

def generate_llm_prompt(scenario):
    """
    Translates the scenario dictionary into the strict natural language 
    prompt used to evaluate the LLM's executive function.
    """
    prompt = (
        f"You are navigating a transport vehicle from '{scenario['start_node']}' to '{scenario['end_node']}'. "
        "Find the fastest valid route without destroying any cargo.\n\n"
    )
    
    # 1. The Mission (Required Items)
    prompt += "### YOUR MISSION\n"
    prompt += "You MUST deliver the following Cargo:\n"
    for item in scenario['required_cargo']:
        prompt += f"- {item['name']} (Weight: {item['weight']})\n"
    prompt += "\n"
    
    # 2. The Armory (Optional Items)
    prompt += "### THE ARMORY\n"
    prompt += "You may choose to bring any of the following protective items:\n"
    for item in scenario['optional_protections']:
        prompt += f"- {item['name']} (Weight: {item['weight']})\n"
    prompt += "\n"
    
    # 3. Vehicle Limits
    prompt += "### VEHICLE LIMIT\n"
    prompt += f"Maximum Total Weight Capacity (Cargo + Protections): {scenario['vehicle_max_weight']}\n\n"
    
    # 4. The Map Translation
    prompt += "### THE MAP\n"
    for node, edges in scenario['graph'].items():
        if not edges:
            continue
        edge_strings = []
        for target, data in edges.items():
            edge_strings.append(f"{target} (Time: {data['time']} hrs, Terrain: {data['terrain']})")
        prompt += f"From {node}, you can travel to: {', '.join(edge_strings)}.\n"
    prompt += "\n"
    
    # 5. The Rules Engine
    prompt += "### THE RULES\n"
    prompt += "1. You cannot exceed the maximum vehicle weight capacity at any time.\n"
    prompt += "2. If you carry 'Glass' through a 'Rocky' terrain, it will shatter.\n"
    prompt += "3. If you carry a 'Hard Drive' through a 'Magnetic' terrain, it will wipe UNLESS you carry a 'Lead Shield'.\n"
    prompt += "4. If you carry 'Ice' through a 'Desert' terrain, it will melt UNLESS you also carry a 'Cooler'.\n"
    prompt += "5. STATE DEGRADATION: Your 'Cooler' only has enough battery to survive exactly ONE 'Desert' terrain. If you enter a second 'Desert' terrain, the Ice will melt.\n"
    prompt += "6. DISTRACTOR RULE: 'Biological Samples' spoil in 'Normal' terrain after 3 hours. (Ensure you are actually carrying these before applying this rule).\n"
    prompt += "7. TIME PENALTY: Carrying the 'Heavy Generator' adds +1 hour to EVERY path you take.\n\n"
    
    # 6. Strict Output Instructions
    prompt += (
        "Provide the absolute fastest valid path. Output ONLY a valid JSON object "
        "with exactly three keys: 'chosen_inventory' (a list of item names you decided to take), "
        "'path' (a list of exact node names), and 'total_time' (an integer representing the total hours)."
    )
    
    return prompt

import random

def generate_random_scenario(num_layers=4, nodes_per_layer=4):
    # 1. Define the item pool
    cargo_pool = [
        {"name": "Ice", "weight": 2},
        {"name": "Hard Drive", "weight": 1},
        {"name": "Glass", "weight": 1},
        {"name": "Biological Samples", "weight": 2}
    ]
    protection_pool = [
        {"name": "Cooler", "weight": 2},
        {"name": "Lead Shield", "weight": 3},
        {"name": "Heavy Generator", "weight": 5}
    ]
    
    # 2. Pick Random Items
    required_cargo = random.sample(cargo_pool, k=random.randint(1, 2))
    optional_protections = random.sample(protection_pool, k=random.randint(2, 3))
    
    # 3. Force the Knapsack Constraint
    base_weight = sum(item['weight'] for item in required_cargo)
    optional_weights = sorted([item['weight'] for item in optional_protections])
    
    # The vehicle can hold the cargo + the lightest protection, but NOT all protections
    if optional_weights:
        max_weight = base_weight + optional_weights[0] + random.randint(0, 1)
        # Ensure it's strictly less than everything combined to force a choice
        total_possible_weight = base_weight + sum(optional_weights)
        if max_weight >= total_possible_weight:
            max_weight = total_possible_weight - 1
    else:
        max_weight = base_weight
        
    # 4. Generate a Random Layered DAG (Directed Acyclic Graph)
    terrains = ["Normal", "Desert", "Rocky", "Magnetic"]
    graph = {"Base": {}, "Destination": {}}
    
    # num_layers = random.randint(2, 4)
    # nodes_per_layer = random.randint(2, 3)
    
    layers = [["Base"]]
    for i in range(num_layers):
        layer_nodes = [f"Node_{i}_{j}" for j in range(nodes_per_layer)]
        for node in layer_nodes:
            graph[node] = {}
        layers.append(layer_nodes)
    layers.append(["Destination"])
    
    # Connect layers
    for i in range(len(layers) - 1):
        current_layer = layers[i]
        next_layer = layers[i+1]
        
        for u in current_layer:
            # Ensure at least one connection to the next layer
            targets = random.sample(next_layer, k=random.randint(1, len(next_layer)))
            for v in targets:
                graph[u][v] = {
                    "time": random.randint(1, 5),
                    "terrain": random.choices(terrains, weights=[25, 25, 25, 25])[0] # Favor Normal slightly
                }

    return {
        "start_node": "Base",
        "end_node": "Destination",
        "required_cargo": required_cargo,
        "optional_protections": optional_protections,
        "vehicle_max_weight": max_weight,
        "graph": graph
    }

def create_dataset(target_size=2):
    dataset = []
    attempts = 0
    print(f"Generating deep scenarios...")
    
    while len(dataset) < target_size:
        attempts += 1
        # Generate a massive 10-node graph
        # number_of_nodes = 15
        scenario = generate_random_scenario(num_layers=5, nodes_per_layer=10)
        solution = solve_knapsack_logistics_puzzle(scenario)
        # deep_path_lebgth  = number_of_nodes//2
        
        if solution["status"] == "Success":
            # Only save puzzles where the path is actually deep (e.g., more than 6 nodes long) an 50% are not NORMAL
            normal_more_than_half = sum(1 for node in solution["optimal_path"] if scenario['graph'][node].get('terrain') == 'Normal') > len(solution["optimal_path"]) / 2
            if len(solution["optimal_path"]) > 0 and not normal_more_than_half:
                dataset_row = {
                    "id": f"fragile_inventory_puzzle_{len(dataset) + 1:03d}",
                    "prompt": generate_llm_prompt(scenario),
                    "expected_output": {
                        "path": solution["optimal_path"],
                        "total_time": solution["optimal_time"],
                        "path_length": len(solution["optimal_path"])
                    },
                    "scenario_details": scenario  # Optional: Include this for debugging or analysis
                }
                dataset.append(dataset_row)
            
    print(f"Generated {target_size} deep scenarios out of {attempts} attempts.")
    return dataset

data = create_dataset(target_size=1)
pprint(data)


Generating deep scenarios...
Generated 1 deep scenarios out of 1 attempts.
[{'expected_output': {'path': ['Base',
                               'Node_0_1',
                               'Node_1_9',
                               'Node_2_9',
                               'Node_3_9',
                               'Node_4_0',
                               'Destination'],
                      'path_length': 7,
                      'total_time': 8},
  'id': 'fragile_inventory_puzzle_001',
  'prompt': "You are navigating a transport vehicle from 'Base' to "
            "'Destination'. Find the fastest valid route without destroying "
            'any cargo.\n'
            '\n'
            '### YOUR MISSION\n'
            'You MUST deliver the following Cargo:\n'
            '- Glass (Weight: 1)\n'
            '\n'
            '### THE ARMORY\n'
            'You may choose to bring any of the following protective items:\n'
            '- Lead Shield (Weight: 3)\n'
            '- Heavy 

In [147]:
solution = solve_knapsack_logistics_puzzle(data[0]["scenario_details"])
pprint(solution)

{'optimal_inventory': ['Glass'],
 'optimal_path': ['Base',
                  'Node_0_1',
                  'Node_1_9',
                  'Node_2_9',
                  'Node_3_9',
                  'Node_4_0',
                  'Destination'],
 'optimal_time': 8,
 'status': 'Success'}


In [148]:
print(data[0]["prompt"])
copy(data[0]["prompt"])

You are navigating a transport vehicle from 'Base' to 'Destination'. Find the fastest valid route without destroying any cargo.

### YOUR MISSION
You MUST deliver the following Cargo:
- Glass (Weight: 1)

### THE ARMORY
You may choose to bring any of the following protective items:
- Lead Shield (Weight: 3)
- Heavy Generator (Weight: 5)
- Cooler (Weight: 2)

### VEHICLE LIMIT
Maximum Total Weight Capacity (Cargo + Protections): 3

### THE MAP
From Base, you can travel to: Node_0_6 (Time: 5 hrs, Terrain: Normal), Node_0_1 (Time: 1 hrs, Terrain: Magnetic), Node_0_2 (Time: 2 hrs, Terrain: Magnetic), Node_0_5 (Time: 2 hrs, Terrain: Desert), Node_0_7 (Time: 3 hrs, Terrain: Magnetic), Node_0_8 (Time: 4 hrs, Terrain: Magnetic), Node_0_3 (Time: 2 hrs, Terrain: Desert), Node_0_0 (Time: 5 hrs, Terrain: Normal), Node_0_9 (Time: 1 hrs, Terrain: Rocky), Node_0_4 (Time: 2 hrs, Terrain: Magnetic).
From Node_0_0, you can travel to: Node_1_0 (Time: 2 hrs, Terrain: Normal), Node_1_7 (Time: 3 hrs, Terrai

In [275]:
def validate_llm_response(scenario, llm_response, ground_truth):
    """
    Evaluates the LLM's generated response against the hard constraints 
    of the scenario and the mathematical ground truth.
    
    Returns a dictionary with 'is_valid', 'is_optimal', and an 'error_reason'
    to help categorize LLM failure modes.
    """
    
    # 1. Basic Format and Key Checks
    try:
        chosen_inventory = llm_response.get("chosen_inventory", [])
        path = llm_response.get("path", [])
        llm_time = llm_response.get("total_time")
    except AttributeError:
        return {"is_valid": False, "is_optimal": False, "error": "Invalid JSON format."}

    if not path or not chosen_inventory or llm_time is None:
        return {"is_valid": False, "is_optimal": False, "error": "Missing required keys."}

    # 2. Inventory (Knapsack) Validation
    required_names = [item['name'] for item in scenario['required_cargo']]
    for req in required_names:
        if req not in chosen_inventory:
            return {"is_valid": False, "is_optimal": False, "error": f"Failed Knapsack: Missing required cargo '{req}'."}

    # Calculate total weight
    all_items = {item['name']: item['weight'] for item in scenario['required_cargo'] + scenario['optional_protections']}
    total_weight = 0
    for item_name in chosen_inventory:
        if item_name not in all_items:
            return {"is_valid": False, "is_optimal": False, "error": f"Hallucinated item: '{item_name}'."}
        total_weight += all_items[item_name]
        
    if total_weight > scenario['vehicle_max_weight']:
        return {"is_valid": False, "is_optimal": False, "error": f"Failed Knapsack: Exceeded max weight ({total_weight} > {scenario['vehicle_max_weight']})."}

    # 3. Endpoint Validation
    if path[0] != scenario['start_node']:
        return {"is_valid": False, "is_optimal": False, "error": "Path did not start at Base."}
    if path[-1] != scenario['end_node']:
        return {"is_valid": False, "is_optimal": False, "error": "Path did not reach Destination."}

    # 4. The Path Simulation (Rules Engine)
    actual_time = 0
    deserts_crossed = 0
    graph = scenario['graph']
    
    has_heavy_gen = "Heavy Generator" in chosen_inventory
    
    for i in range(len(path) - 1):
        current_node = path[i]
        next_node = path[i+1]
        
        # Check if edge exists (Hallucination check)
        if current_node not in graph or next_node not in graph[current_node]:
            return {"is_valid": False, "is_optimal": False, "error": f"Hallucinated edge: {current_node} -> {next_node} does not exist."}
            
        edge_data = graph[current_node][next_node]
        terrain = edge_data['terrain']
        base_time = edge_data['time']
        
        # Terrain Constraints
        if terrain == "Desert":
            deserts_crossed += 1
            if "Ice" in chosen_inventory:
                if "Cooler" not in chosen_inventory:
                    return {"is_valid": False, "is_optimal": False, "error": "Constraint Failed: Ice melted in Desert (No Cooler)."}
                if deserts_crossed > 1:
                    return {"is_valid": False, "is_optimal": False, "error": "Constraint Failed: Ice melted (Cooler battery died on 2nd Desert)."}
                    
        if terrain == "Rocky" and "Glass" in chosen_inventory:
            return {"is_valid": False, "is_optimal": False, "error": "Constraint Failed: Glass shattered in Rocky terrain."}
            
        if terrain == "Magnetic" and "Hard Drive" in chosen_inventory:
            if "Lead Shield" not in chosen_inventory:
                return {"is_valid": False, "is_optimal": False, "error": "Constraint Failed: Hard Drive wiped in Magnetic terrain (No Shield)."}
                
        # Time calculation
        actual_time += base_time + (1 if has_heavy_gen else 0)

    # 5. Math & Optimality Check
    if actual_time != llm_time:
        return {"is_valid": False, "is_optimal": False, "error": f"Math Error: LLM claimed {llm_time} hrs, but actual valid path takes {actual_time} hrs."}
        
    if actual_time > ground_truth['optimal_time']:
        return {
            "is_valid": True, 
            "is_optimal": False, 
            "error": f"Sub-optimal: Path is valid ({actual_time} hrs), but optimal is {ground_truth['optimal_time']} hrs."
        }
        
    return {
        "is_valid": True, 
        "is_optimal": True, 
        "error": None,
        "message": "Perfect Evaluation! LLM successfully matched ground truth."
    }


test_solution = {
    "chosen_inventory": solution.get("optimal_inventory", []),
    "path": solution.get("optimal_path", []),
    "total_time": solution.get("optimal_time", 0)
}
llm_response_1 = {
  "chosen_inventory": ["Ice", "Biological Samples"],
  "path": ["Base", "Node_0_1", "Node_1_1", "Node_2_1", "Destination"],
  "total_time": 12
}

is_valid = validate_llm_response(random_scenario, llm_response_1, solution)
pprint(is_valid)

{'error': 'Sub-optimal: Path is valid (12 hrs), but optimal is 11 hrs.',
 'is_optimal': False,
 'is_valid': True}


In [245]:
example_scenario = {
    "start_node": "Base",
    "end_node": "Destination",
    "required_cargo": [
        {"name": "Ice", "weight": 2},
        {"name": "Hard Drive", "weight": 1}
    ],
    "optional_protections": [
        {"name": "Cooler", "weight": 2},
        {"name": "Lead Shield", "weight": 3}
    ],
    "vehicle_max_weight": 6, 
    # NOTE: Required Cargo = 3. 
    # Remaining Capacity = 3. 
    # You can take the Cooler (2) OR the Lead Shield (3), but taking BOTH equals 8 (Exceeds 6).
    
    "graph": {
        "Base": {
            "Node_1": {"time": 2, "terrain": "Magnetic"},
            "Node_2": {"time": 1, "terrain": "Desert"},
            "Node_4": {"time": 3, "terrain": "Desert"},
            "Node_5": {"time": 4, "terrain": "Normal"}
        },
        "Node_1": {
            "Destination": {"time": 2, "terrain": "Normal"}
        },
        "Node_2": {
            "Node_3": {"time": 1, "terrain": "Desert"}
        },
        "Node_3": {
            "Destination": {"time": 1, "terrain": "Normal"}
        },
        "Node_4": {
            "Destination": {"time": 2, "terrain": "Normal"}
        },
        "Node_5": {
            "Destination": {"time": 4, "terrain": "Normal"}
        },
        "Destination": {}
    }
}

solution = solve_knapsack_logistics_puzzle(example_scenario)
print("Optimal Solution:", solution)

Optimal Solution: {'status': 'Success', 'optimal_time': 4, 'optimal_path': ['Base', 'Node_1', 'Destination'], 'optimal_inventory': ['Ice', 'Hard Drive', 'Lead Shield']}
